In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')


In [6]:
def engineer_netflix_features(dataset_path):
    """
    Loads raw Netflix data, cleans missing targets, maps audience demographics,
    and engineers features for machine learning.
    """
    # Load dataset
    raw_data = pd.read_csv(dataset_path)
    
    # Drop records missing the target variable
    clean_df = raw_data.dropna(subset=['rating']).copy()
    
    # Define custom mapping for audience demographics
    demographic_mapping = {
        'TV-MA': 'Mature', 'R': 'Mature', 'NC-17': 'Mature', 'UR': 'Mature', 'NR': 'Mature',
        'TV-14': 'Teen', 'PG-13': 'Teen',
        'TV-PG': 'Older Children', 'TV-Y7': 'Older Children', 'TV-Y7-FV': 'Older Children', 'PG': 'Older Children',
        'TV-Y': 'Toddlers/Kids', 'G': 'Toddlers/Kids'
    }
    
    # Apply mapping and drop any unmapped edge cases
    clean_df['audience_class'] = clean_df['rating'].map(demographic_mapping)
    clean_df = clean_df.dropna(subset=['audience_class'])
    
    # Feature 1: Format Duration (Extract integers from "90 min" or "2 Seasons")
    clean_df['runtime_val'] = clean_df['duration'].str.extract(r'(\d+)').astype(float)
    clean_df['runtime_val'] = clean_df['runtime_val'].fillna(clean_df['runtime_val'].median())
    
    # Feature 2: Binary Encode Content Type
    clean_df['is_series'] = (clean_df['type'] == 'TV Show').astype(int)
    
    # Feature 3: One-Hot Encode Genres (listed_in)
    # Split strings into lists of genres, stripping whitespace
    genre_lists = clean_df['listed_in'].apply(lambda item: [g.strip() for g in item.split(',')])
    
    # Binarize genres
    ml_binarizer = MultiLabelBinarizer()
    genre_matrix = pd.DataFrame(
        ml_binarizer.fit_transform(genre_lists), 
        columns=ml_binarizer.classes_, 
        index=clean_df.index
    )
    
    # Assemble final feature matrix (X) and target vector (y)
    # Including release_year as a numerical feature
    core_features = clean_df[['is_series', 'release_year', 'runtime_val']]
    X_features = pd.concat([core_features, genre_matrix], axis=1)
    y_target = clean_df['audience_class']
    
    return X_features, y_target

# EXECUTION:
# Replace 'netflix_titles.csv' with your actual file path
X, y = engineer_netflix_features('./data/raw/Dataset.csv')

# Split the data into training and testing sets (80/20 split), maintaining class proportions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101, stratify=y)


In [7]:
def optimize_tree_models(features_train, targets_train):
    """
    Performs grid search cross-validation to find the optimal hyperparameters
    for both a Decision Tree and a Random Forest classifier.
    """
    print("Initiating Decision Tree Optimization...")
    
    # Define hyperparameter grid for Decision Tree
    tree_search_space = {
        'max_depth': [5, 12, 20, None],
        'min_samples_leaf': [2, 5, 10],
        'criterion': ['gini', 'entropy']
    }
    
    base_tree = DecisionTreeClassifier(random_state=101)
    optimized_tree = GridSearchCV(
        estimator=base_tree, 
        param_grid=tree_search_space, 
        cv=3, 
        scoring='accuracy', 
        n_jobs=-1
    )
    optimized_tree.fit(features_train, targets_train)
    
    print("Initiating Random Forest Optimization...")
    
    # Define hyperparameter grid for Random Forest
    forest_search_space = {
        'n_estimators': [50, 150, 250],
        'max_depth': [10, 20, None],
        'bootstrap': [True]
    }
    
    base_forest = RandomForestClassifier(random_state=101)
    optimized_forest = GridSearchCV(
        estimator=base_forest, 
        param_grid=forest_search_space, 
        cv=3, 
        scoring='accuracy', 
        n_jobs=-1
    )
    optimized_forest.fit(features_train, targets_train)
    
    return optimized_tree, optimized_forest

# EXECUTION:
best_dt_model, best_rf_model = optimize_tree_models(X_train, y_train)


Initiating Decision Tree Optimization...
Initiating Random Forest Optimization...


In [8]:
def generate_performance_report(trained_model, model_name, features_test, targets_test):
    """
    Generates predictions on test data and prints comprehensive evaluation metrics.
    """
    print(f"========== EVALUATION: {model_name.upper()} ==========")
    
    # Extract the best hyperparameters found during GridSearch
    print(f"Optimal Hyperparameters: {trained_model.best_params_}\n")
    
    # Generate predictions
    predictions = trained_model.predict(features_test)
    
    # Calculate overarching metrics
    overall_accuracy = accuracy_score(targets_test, predictions)
    print(f"Overall Test Accuracy: {overall_accuracy * 100:.2f}%\n")
    
    # Detailed classification metrics per category
    print("Detailed Classification Report:")
    print(classification_report(targets_test, predictions))
    print("======================================================\n")

# EXECUTION:
generate_performance_report(best_dt_model, "Decision Tree", X_test, y_test)
generate_performance_report(best_rf_model, "Random Forest", X_test, y_test)


========== EVALUATION: DECISION TREE ==========
Optimal Hyperparameters: {'criterion': 'gini', 'max_depth': 12, 'min_samples_leaf': 5}

Overall Test Accuracy: 58.87%

Detailed Classification Report:
                precision    recall  f1-score   support

        Mature       0.62      0.80      0.70       818
Older Children       0.71      0.36      0.47       297
          Teen       0.48      0.40      0.44       530
 Toddlers/Kids       0.59      0.49      0.54        69

      accuracy                           0.59      1714
     macro avg       0.60      0.51      0.54      1714
  weighted avg       0.59      0.59      0.57      1714


========== EVALUATION: RANDOM FOREST ==========
Optimal Hyperparameters: {'bootstrap': True, 'max_depth': 10, 'n_estimators': 150}

Overall Test Accuracy: 62.31%

Detailed Classification Report:
                precision    recall  f1-score   support

        Mature       0.62      0.91      0.73       818
Older Children       0.72      0.39      